## Load the summary information and preprocessed data

In [1]:
import os
import json
import pandas as pd

from pathlib import Path

from single_sample_gsea import ss_gsea


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 81 samples and 11899 columns.
First few rows of the data:


,New OS,vital.status NEW,New RemDur,event,Patient_ID,Cluster_Definition_Post_FAB_Correction,Batches,Platform-RNASeq,Platform-FM1,Platform-RPPA,...,COX2,ATP8,ATP6,COX3,ND3,ND4L,ND4,ND5,ND6,CYTB
0,84.00,1,21.14,1,3003486-N,Cluster_3,Batch1,RNASeq,FM1,RPPA,...,2.226803,1.146102,2.057132,1.723706,1.219951,1.308459,2.791821,2.582067,1.570612,1.833584
1,8.29,1,NaN,0,3044356-N,Cluster_2,Batch1,RNASeq,FM1,RPPA,...,4.098751,2.785802,4.125153,4.457178,2.766367,2.800134,4.732233,4.073847,2.629900,4.112465
2,22.43,1,NaN,0,3091926-N,Cluster_1,Batch1,RNASeq,FM1,RPPA,...,4.953971,3.637522,4.966192,5.198860,3.386643,3.627556,5.615686,4.996264,3.570893,5.019097
3,129.29,1,111.71,1,3103988-N,Cluster_2,Batch1,RNASeq,FM1,RPPA,...,3.943428,2.473518,3.655212,3.787883,2.247079,2.565096,4.512628,4.068320,2.731174,3.711038
4,36.57,1,NaN,0,3118298-N,Cluster_3,Batch1,RNASeq,FM1,RPPA,...,3.415620,1.849062,3.000872,2.935574,1.845778,1.992381,3.951988,3.377248,1.986683,3.110679


## Load gene sets

In [2]:
MSIGDB_JSON = Path("h.all.v2026.1.Hs.json")

with MSIGDB_JSON.open(encoding="utf-8") as f:
    gene_sets = {name: data["geneSymbols"] for name, data in json.load(f).items()}

print(f"Loaded {len(gene_sets)} gene sets.")


Loaded 50 gene sets.


## Compute single sample GSEA on the preprocessed data

Select only gene columns for the computation.

In [3]:
# IMPORTANT: This is dataset-specific and is the only variable that needs to be changed for each of the dataset.
sample_id_column = 'sample_title'

gene_expression_df = df.iloc[:, summary_data['data_summary']['clinical_features']:].copy()
gene_expression_df.index = df[sample_id_column].astype(str)
gene_expression_df = gene_expression_df.T
gene_expression_df.index.name = "gene"

result = ss_gsea(gene_expression_df, gene_sets)


## Merge the results with the clinical data

The output table should have the same number of rows as before and columns should sum up to clinical data + 50 gene sets.

In [4]:
clinical_df = df.iloc[:, :summary_data["data_summary"]["clinical_features"]].copy()
clinical_df = clinical_df.set_index(sample_id_column)

output_df = pd.concat([clinical_df, result], axis=1)
output_df = output_df.reset_index(drop=False)

output_df.to_csv(f"{GEO_ID}_preprocessed_ssgsea.csv", index=False)

# sanity check (there is 50 gene sets)
assert output_df.shape[1] == summary_data["data_summary"]["clinical_features"] + 50
